# SFUMATO uncertainty-aware spatial domains

SFUMATO discovers expression domains at user-selected physical scales and retains posterior uncertainty. The resulting labels describe **domains or niches**, not individual cells.

Input preparation below is CPU-only. The Bayesian Gaussian mixture stage requires a CUDA GPU. See `TRANSCRIPT_ANALYSIS.md` for the tested SFUMATO revision and environment setup.

In [ ]:
from pathlib import Path
import json
import subprocess
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc

import ISS_postprocessing.spatial_domains as DOMAINS

## Dataset, SFUMATO checkout, and scale

Decoded pixel coordinates are converted to micrometres before SFUMATO runs, so `bin_width_um` has an explicit physical meaning.

In [ ]:
input_dir = Path("/path/to/experiment")
region = "R1"
pixel_size_um = 0.325
coordinate_unit = "pixel"
sfumato_root = Path("/path/to/SFUMATO")

transcripts = input_dir / region / "decoding" / "2_decoded" / f"{region}_decoded.csv"
spots = pd.read_csv(transcripts)
display(spots.head())
print(f"{len(spots):,} transcripts; {spots['target'].nunique():,} genes")

## Choose a scale and prepare the run

For cell-scale exploration, start near a typical cell diameter and use a low density cutoff. For niche-scale analysis, increase `bin_width_um` and `smoothing_factor`. `svd_components=None` automatically selects at most 10 components while respecting a small ISS panel.

In [ ]:
parameters = dict(
    k_list=[8, 12, 16],
    bin_width_um=10,           # current SFUMATO requires a whole number
    smoothing_factor=5,
    min_transcripts_per_bin=4,
    svd_components=None,
    bgm_oversample=1.5,
    seed=8,
    save_transcript_probabilities=False,
)

prepared = DOMAINS.prepare_sfumato_analysis(
    transcripts,
    input_dir=input_dir,
    region=region,
    coordinate_unit=coordinate_unit,
    pixel_size_um=pixel_size_um,
    **parameters,
)
print(json.dumps(json.loads(prepared['config'].read_text()), indent=2))

In [ ]:
# Confirm that the external environment sees CUDA before starting a long run.
gpu_check = subprocess.run(
    ["conda", "run", "-n", "sfumato", "python", "-c",
     "import torch; print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No CUDA GPU')"],
    capture_output=True, text=True, check=False,
)
print(gpu_check.stdout)
print(gpu_check.stderr)

## Run SFUMATO on CUDA

The adapter preserves SFUMATO's complete native output and additionally creates one standardized bin-level AnnData for every requested `K`. Revision mismatches are reported; set `strict_revision=True` for fully pinned production runs.

In [ ]:
outputs = DOMAINS.run_sfumato_analysis(
    transcripts,
    input_dir=input_dir,
    region=region,
    coordinate_unit=coordinate_unit,
    pixel_size_um=pixel_size_um,
    sfumato_root=sfumato_root,
    sfumato_python=["conda", "run", "-n", "sfumato", "python"],
    strict_revision=False,
    overwrite=False,
    **parameters,
)
outputs

## Compare resolutions and uncertainty

`cluster` is the dominant domain, `p1` and `p2` are the top two posterior probabilities, and `compl_p1 = 1 - p1` highlights ambiguous or transitional locations. Domain numbers are not cell types until annotated biologically.

In [ ]:
result_keys = sorted(key for key in outputs if key.startswith('bins_K'))
fig, axes = plt.subplots(len(result_keys), 2, figsize=(12, 5 * len(result_keys)), squeeze=False)

for row, key in enumerate(result_keys):
    adata = sc.read_h5ad(outputs[key])
    xy = adata.obsm['spatial_um']
    hard = axes[row, 0].scatter(xy[:, 0], xy[:, 1], c=adata.obs['cluster'].astype(int), s=3, cmap='tab20')
    axes[row, 0].set_title(f"{key}: dominant domain")
    uncertain = axes[row, 1].scatter(xy[:, 0], xy[:, 1], c=adata.obs['compl_p1'], s=3, cmap='magma', vmin=0, vmax=1)
    axes[row, 1].set_title(f"{key}: uncertainty (1 - p1)")
    fig.colorbar(uncertain, ax=axes[row, 1])
    for ax in axes[row]:
        ax.set_aspect('equal')
        ax.invert_yaxis()
        ax.set_xlabel('x (µm)')
        ax.set_ylabel('y (µm)')
plt.tight_layout();